# P03A — Generator data sintesis beton

Seed baku 30303; 1.200 spesimen virtual. Rumus DGP dibuka agar latihan dapat diulang. Nilai bukan hasil uji, tidak mewakili populasi beton, dan tidak boleh digunakan untuk menentukan campuran.

**Cara menjalankan**: di Colab atau komputer lokal, jalankan sel di bawah sekali. Berkas `data_sintesis_beton.csv` (untuk Orange dan Python) dan `data_sintesis_beton_orange.tab` (dengan peran kolom sudah ditetapkan) ditulis ke `praktikum/bab03/` relatif terhadap folder kerja. Di Colab, folder kerjanya `/content`; unduh kedua berkas itu bila akan dibuka di Orange. Lanjutkan ke P03B pada sesi yang sama.

In [ ]:
# Folder kerja: jalankan dari folder mana pun. Sel ini pindah ke akar repositori agar jalur data relatif (praktikum/...) benar.
import os
from pathlib import Path
_akar = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "praktikum").is_dir() and (p / "notebooks").is_dir()), None)
if _akar is not None:
    os.chdir(_akar)
else:
    print("Akar repositori tidak ditemukan. Buka notebook dari dalam folder repositori hasil clone.")

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
SEED=30303; rng=np.random.default_rng(SEED); out=Path('praktikum/bab03'); out.mkdir(parents=True,exist_ok=True)
n=1200
cement=rng.uniform(250,480,n); wc=rng.uniform(.30,.82,n); water=cement*wc
fly=rng.uniform(0,120,n); sp=rng.uniform(0,18,n); coarse=rng.uniform(850,1150,n); fine=rng.uniform(600,900,n); age=rng.choice([3,7,14,28,56,90],n,p=[.12,.18,.15,.28,.12,.15])
noise=rng.normal(0,3.0,n)
# Persamaan hanya membangkitkan pola latihan; bukan model campuran atau hukum material.
strength=8 + .08*(cement-250) - 42*(wc-.45) + .045*fly + 2.5*np.log1p(age) + .10*sp + .018*(1100-coarse) + .012*(800-fine) + noise
strength=np.clip(strength,5,None)
df=pd.DataFrame({'id_spesimen':[f'S{i:04d}' for i in range(1,n+1)],'semen_kg_m3':cement.round(1),'air_kg_m3':water.round(1),'rasio_air_semen':wc.round(3),'fly_ash_kg_m3':fly.round(1),'superplasticizer_kg_m3':sp.round(2),'agregat_kasar_kg_m3':coarse.round(1),'agregat_halus_kg_m3':fine.round(1),'umur_hari':age,'kuat_tekan_MPa':strength.round(2)})
csv=out/'data_sintesis_beton.csv';df.to_csv(csv,index=False)
cols=list(df.columns);types=['string']+['continuous']*(len(cols)-1);flags=['meta']+['']*(len(cols)-2)+['class']
with (out/'data_sintesis_beton_orange.tab').open('w',encoding='utf-8',newline='') as f:
 f.write('\t'.join(cols)+'\n'+'\t'.join(types)+'\n'+'\t'.join(flags)+'\n');df.to_csv(f,sep='\t',index=False,header=False)
formula='strength = max(5, 8 + 0.08(cement-250) - 42(w/c-0.45) + 0.045(fly ash) + 2.5 ln(1+age) + 0.10(superplasticizer) + 0.018(1100-coarse) + 0.012(800-fine) + Normal(0,3))'
print('seed',SEED,'shape',df.shape,'ID unique',df.id_spesimen.nunique(),'missing',int(df.isna().sum().sum()),'exact_duplicates',int(df.duplicated().sum()))
print('DGP:',formula);print(df.describe().round(2).to_string());print('TRAINING DATA ONLY: synthetic, not field measurements or mix-design advice')

seed 30303 shape (1200, 10) ID unique 1200 missing 0 exact_duplicates 0
DGP: strength = max(5, 8 + 0.08(cement-250) - 42(w/c-0.45) + 0.045(fly ash) + 2.5 ln(1+age) + 0.10(superplasticizer) + 0.018(1100-coarse) + 0.012(800-fine) + Normal(0,3))
       semen_kg_m3  air_kg_m3  rasio_air_semen  fly_ash_kg_m3  superplasticizer_kg_m3  agregat_kasar_kg_m3  agregat_halus_kg_m3  umur_hari  kuat_tekan_MPa
count      1200.00    1200.00          1200.00        1200.00                 1200.00              1200.00              1200.00    1200.00         1200.00
mean        365.38     204.44             0.56          60.11                    9.25               999.23               748.09      31.37           26.44
std          64.98      65.49             0.15          35.14                    5.20                87.97                85.54      29.21            9.17
min         250.20      78.00             0.30           0.00                    0.07               850.10               600.30       3.0